# Databricks pharmacy pipeline

Orquesta **SQL Warehouse** de Databricks y el catálogo Unity **`pharmacy`**.

**Qué hace**
1. Crea esquemas `bronze` / `silver` / `gold` y tablas bronze (Delta) desde los `.sql` modulares.
2. Carga los CSV de `data/raw_data/` en las tablas **`pharmacy.bronze.raw_dim_*`** (las columnas que coincidan con el esquema bronze; `_ingest_*` quedan por defecto).
3. Ejecuta silver → gold → vistas (CTAS / `CREATE VIEW`) para que dimensiones curadas reflejen los datos cargados.

**Hechos (facts)** en bronze **no** se cargan desde estos CSV: el modelo bronze de hechos no coincide con las exportaciones de SQL Server. Los hechos en silver/gold quedarán vacíos hasta un ETL aparte (Spark, `COPY INTO`, etc.).

**Prerrequisitos**
- Catálogo `pharmacy` en Unity Catalog (y permisos de DDL). El script intenta `CREATE CATALOG IF NOT EXISTS pharmacy`; si falla, créalo en la UI.
- Variables en `ai-sql-migration/.env`: `DATABRICKS_HOST`, `DATABRICKS_TOKEN`, `DATABRICKS_WAREHOUSE_ID`.
- Ejecutar el notebook con el kernel del proyecto (`uv run python` / entorno donde esté `databricks-sql-connector` y `pandas`).

## Probar conexión (mínimo)

- **Terminal** (desde la carpeta `ai-sql-migration`): `uv run python scripts/test_databricks_sql_connection.py`
- **Notebook**: ejecuta la celda de código **siguiente** después de la que define `ROOT` y llama a `load_dotenv`.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def find_repo_root() -> Path:
    """Locate ai-sql-migration (folder with pyproject.toml and data/init_db_databricks.py)."""
    cwd = Path.cwd().resolve()
    for p in (cwd, cwd.parent, cwd / "ai-sql-migration"):
        if (p / "pyproject.toml").is_file() and (p / "data" / "init_db_databricks.py").is_file():
            return p
    raise RuntimeError(
        "Set the notebook working directory to ai-sql-migration, or open the repo from its root."
    )


ROOT = find_repo_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv

load_dotenv(ROOT / ".env", override=True)
print("Repo:", ROOT)

In [ ]:
# Misma forma que la documentación: databricks.sql.connect + SELECT 1
# Requiere haber ejecutado la celda anterior (load_dotenv).
import os
from urllib.parse import urlparse
from databricks import sql

host_raw = os.environ.get("DATABRICKS_HOST", "").strip().rstrip("/")
token = os.environ.get("DATABRICKS_TOKEN", "").strip()
warehouse_id = os.environ.get("DATABRICKS_WAREHOUSE_ID", "").strip()

if not host_raw or not token or not warehouse_id:
    raise RuntimeError("Faltan DATABRICKS_HOST, DATABRICKS_TOKEN o DATABRICKS_WAREHOUSE_ID (¿ejecutaste load_dotenv?)")

if host_raw.startswith("http://") or host_raw.startswith("https://"):
    server_hostname = urlparse(host_raw).hostname
else:
    server_hostname = host_raw.replace("https://", "").replace("http://", "")

http_path = f"/sql/1.0/warehouses/{warehouse_id}"

print("server_hostname:", server_hostname)
print("http_path:", http_path)
print("token length:", len(token))

with sql.connect(
    server_hostname=server_hostname,
    http_path=http_path,
    access_token=token,
) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT 1 AS ok")
        row = cur.fetchone()

print("Query result:", row)
print("Conexión OK.")

In [ ]:
from data import init_db_databricks as dbp

# Full pipeline: bronze DDL → load dimensions → silver / gold / views
# Opcional: max_rows_per_table=100 o DATABRICKS_MAX_ROWS_PER_TABLE en .env
dbp.init(data_path="raw_data", skip_ensure_catalog=False)

## Pasos opcionales

Equivalente CLI desde `ai-sql-migration`:

- `uv run python data/init_db_databricks.py ddl-only` — solo DDL completo (bronze+silver+gold+views) sin cargar CSV.
- `uv run python data/init_db_databricks.py load-dims` — solo truncar + cargar dimensiones bronze (requiere tablas ya creadas). Añade `--max-rows 100` para limitar filas por tabla.
- `uv run python data/init_db_databricks.py layer-counts` — conteo de tablas por capa (`schemas/layer_counts.sql`).

In [ ]:
# Descomenta para inspeccionar conteos por capa sin re-ejecutar todo el pipeline
# with dbp.get_connection() as conn:
#     print(dbp.run_layer_counts(conn))